In [1]:
import matminer
import numpy as np
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, f1_score, precision_score, recall_score

In [2]:
df_sc = pd.read_csv("df_sc_ef_ElementFraction_ftd.csv").dropna()
df_sc

FileNotFoundError: [Errno 2] No such file or directory: 'df_sc_ef_ElementFraction_ftd.csv'

In [3]:
df_sc = df_sc[df_sc["Critical Temp"]>=10]
df_sc

,composition,Critical Temp,_Composition,H,He,Li,Be,B,C,N,...,Pu,Am,Cm,Bk,Cf,Es,Fm,Md,No,Lr
0,Ba0.4K0.6Fe2As2,31.20,Ba0.4 K0.6 Fe2 As2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Ca0.4Ba1.25La1.25Cu3O6.98,40.10,Ca0.4 Ba1.25 La1.25 Cu3 O6.98,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,La1.71Sr0.29Cu0.94Co0.06O4,33.00,La1.71 Sr0.29 Cu0.94 Co0.06 O4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10,Nb3Sn0.85Tl0.15,18.20,Nb3 Sn0.85 Tl0.15,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
11,Pb0.5Cu0.5Sr0.9La1.1Cu1O5.16,28.10,Pb0.5 Cu1.5 Sr0.9 La1.1 O5.16,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16400,Dy1Ba2Cu2.8Zn0.2O6.95,13.00,Dy1 Ba2 Cu2.8 Zn0.2 O6.95,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
16401,Bi2Ca2.5Sm0.5Cu2O8.33,22.00,Bi2 Ca2.5 Sm0.5 Cu2 O8.33,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
16408,La1.78Sr0.22Cu0.9975Zn0.0025O4,19.25,La1.78 Sr0.22 Cu0.9975 Zn0.0025 O4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
16411,Pb2Sr2Ho0.5Ca0.5Cu2.982Al0.018O8,63.60,Pb2 Sr2 Ho0.5 Ca0.5 Cu2.982 Al0.018 O8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
X = df_sc.iloc[:, 3:]
y = df_sc['Critical Temp']

In [5]:
df_sc['Critical Temp']

0        31.20
1        40.10
6        33.00
10       18.20
11       28.10
         ...  
16400    13.00
16401    22.00
16408    19.25
16411    63.60
16412    34.80
Name: Critical Temp, Length: 6248, dtype: float64

In [6]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [7]:
X_train.shape

(4686, 103)

In [8]:
# Build the model
model = RandomForestRegressor()

# Hyperparameter tuning using GridSearchCV
param_grid = {
    'n_estimators': [50, 100, 200, 300],
    'max_depth': [None, 10, 20, 30, 50],
    'min_samples_split': [2, 3, 5, 7],
    'min_samples_leaf': [1, 2, 3, 4],
    'bootstrap': [True, False]
}

In [ ]:
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1, verbose=2)
grid_search.fit(X_train, y_train)

# Evaluate the model
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

Fitting 5 folds for each of 640 candidates, totalling 3200 fits


In [ ]:
# Print evaluation metrics
print("Mean Squared Error (MSE):", mean_squared_error(y_test, y_pred))
print("R-squared:", r2_score(y_test, y_pred))
print("Mean Absolute Error (MAE):", mean_absolute_error(y_test, y_pred))
print("Root Mean Squared Error (RMSE):", np.sqrt(mean_squared_error(y_test, y_pred)))
print("Adjusted R-squared:", r2_score(y_test, y_pred, multioutput='uniform_average'))

In [ ]:
pd.DataFrame(grid_search.cv_results_)

In [ ]:
filename = 'sc_ef_best_model_yet.pkl'
with open(filename, 'wb') as f:
    pickle.dump(best_model, f)

In [ ]:
# # Example: Create a new data point (features) for prediction
# new_data_point = [[feature1_value, feature2_value, ...]]

# # Predict using the loaded model
# predicted_value = loaded_model.predict(new_data_point)

# print("Predicted value:", predicted_value)


In [ ]:
# Mean Squared Error (MSE): 285.50798228513975
# R-squared: 0.5568746119199605
# Mean Absolute Error (MAE): 10.705040122111116
# Root Mean Squared Error (RMSE): 16.896981454838013
# Adjusted R-squared: 0.5568746119199605


# On 9000 rows
# Mean Squared Error (MSE): 138.87230308323188
# R-squared: 0.8021012250455234
# Mean Absolute Error (MAE): 6.131617541973627
# Root Mean Squared Error (RMSE): 11.784409322627583
# Adjusted R-squared: 0.8021012250455234

# Mean Squared Error (MSE): 127.29734257361541
# R-squared: 0.827537081963444
# Mean Absolute Error (MAE): 5.243760973322503
# Root Mean Squared Error (RMSE): 11.282612400220767
# Adjusted R-squared: 0.827537081963444

# EF_filtered
# Mean Squared Error (MSE): 104.16987815449859
# R-squared: 0.8788833654783211
# Mean Absolute Error (MAE): 6.1709604490213135
# Root Mean Squared Error (RMSE): 10.206364590514028
# Adjusted R-squared: 0.8788833654783211